# Predicción de planes móviles de Megaline

La compañía móvil Megaline ha detectado que muchos de sus clientes continúan utilizando planes heredados y busca migrarlos hacia uno de sus planes actuales: Smart o Ultra.

Para apoyar este proceso, se desarrollará un modelo de machine learning capaz de analizar el comportamiento mensual de los usuarios y recomendar el plan más adecuado. Para ello, se utilizarán datos de clientes que ya utilizan los nuevos planes, incluyendo información sobre llamadas, minutos de uso, mensajes y consumo de Internet.

El objetivo es construir un modelo de clasificación que permita predecir si un usuario debería utilizar el plan Smart o Ultra, alcanzando una exactitud mínima de 0.75.

## Introducción
En este proyecto se analizará un conjunto de datos que contiene información sobre el comportamiento mensual de clientes de Megaline que ya utilizan los planes Smart o Ultra. Las características disponibles incluyen el número de llamadas realizadas, la duración total de las llamadas, la cantidad de mensajes enviados y el tráfico de Internet utilizado.

En primer lugar, se examinará la estructura y calidad de los datos para comprobar que sean adecuados para el entrenamiento de modelos de machine learning. Posteriormente, los datos se dividirán en conjuntos de entrenamiento, validación y prueba, permitiendo entrenar los modelos, ajustar sus hiperparámetros y evaluar finalmente su capacidad para generalizar a nuevas observaciones.

Se investigarán distintos algoritmos de clasificación, como árboles de decisión, bosques aleatorios y regresión logística, comparando su exactitud para seleccionar el modelo con mejor desempeño. Finalmente, el modelo seleccionado se evaluará utilizando el conjunto de prueba y se realizará una prueba de cordura para comprobar que sus resultados sean superiores a una estrategia de referencia simple.

Me gusta esta introducción porque no promete resultados que todavía no conocemos; simplemente explica qué vamos a hacer y por qué.

## Inicialización

In [1]:
# Cargar las librerías
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Cargar datos

In [2]:
# Cargar el dataset
df = pd.read_csv('datasets/users_behavior.csv')

## Explorar datos

In [3]:
# Verificamos que los datos fueron procesados.
print("Informacion de las columnas")
df.info()
print("Datos ausentes:\n", df.isna().sum())
print("Datos duplicados:", df.duplicated().sum())
print("Resumen estadístico:")
print(df.describe())
print("Dimensiones del dataset:", df.shape)

Informacion de las columnas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
Datos ausentes:
 calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64
Datos duplicados: 0
Resumen estadístico:
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100
min       0.000000     0.000000     0.000000      0.000000     0.000000
25%      40.000000   274.575000     9.00

In [4]:
# Distribución de la variable objetivo

print(df['is_ultra'].value_counts())
print(df['is_ultra'].value_counts(normalize=True))



is_ultra
0    2229
1     985
Name: count, dtype: int64
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64


El conjunto de datos contiene 3214 observaciones y 5 columnas. No se encontraron valores ausentes ni registros duplicados, por lo que no es necesario realizar una etapa adicional de limpieza antes del modelado.

Las cuatro variables calls, minutes, messages y mb_used representan características del comportamiento mensual de los usuarios, mientras que is_ultra será la variable objetivo para la tarea de clasificación, messages se encuentra almacenada como tipo flotante, aunque representa conteos enteros. Dado que los valores son consistentes y el dataset ya fue procesado, no es necesario modificar su tipo para el modelado.

La revisión de los valores mínimos y máximos no muestra valores evidentemente incorrectos. Existen usuarios con valores de cero en algunas variables de uso, lo cual es posible si durante el mes no utilizaron determinado servicio. Los valores máximos son considerablemente superiores a los promedios, pero se mantienen dentro de rangos plausibles y no existen motivos suficientes para tratarlos como errores.

La variable objetivo no se encuentra distribuida de manera uniforme: 2229 observaciones corresponden al plan Smart (0) y 985 al plan Ultra (1), aproximadamente un 69 % y 31 %, respectivamente. Esta distribución deberá tenerse en cuenta al evaluar el modelo, ya que una estrategia simple que predijera siempre la clase mayoritaria alcanzaría una exactitud cercana al 69 %.

El tamaño total del dataset también es relevante al dividirlo en conjuntos de entrenamiento, validación y prueba. Dado que es necesario disponer de suficientes datos tanto para ajustar los hiperparámetros como para realizar una evaluación final confiable, se utilizará una proporción 60/20/20, manteniendo la mayor parte de los datos para entrenamiento sin reducir demasiado los conjuntos de validación y prueba.

## Preparación de los datos

Para entrenar y evaluar los modelos, se separará la variable objetivo is_ultra de las características utilizadas para realizar las predicciones.
Posteriormente, los datos se dividirán en tres conjuntos:
- 60 % para entrenamiento.
- 20 % para validación.
- 20 % para prueba.

El conjunto de entrenamiento se utilizará para ajustar los modelos, el conjunto de validación para comparar algoritmos y seleccionar hiperparámetros, y el conjunto de prueba se reservará para evaluar únicamente el modelo final.

Debido a que la variable objetivo presenta una distribución aproximada de 69 % para Smart y 31 % para Ultra, se utilizará una división estratificada para conservar aproximadamente esta proporción en los tres conjuntos.

In [5]:
# Separar características y variable objetivo
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Separar 60 % para entrenamiento y 40 % para validación + prueba
features_train, features_temp, target_train, target_temp = train_test_split(features, target, test_size=0.40, random_state=12345, stratify=target)
# Dividir el conjunto temporal en validación y prueba
features_valid, features_test, target_valid, target_test = train_test_split(features_temp, target_temp, test_size=0.50, random_state=12345, stratify=target_temp)

In [6]:
# Verificamos los tamaños correctos
print('Conjunto de entrenamiento:')
print(features_train.shape)
print(target_train.shape)
print('Distribución del conjunto de entrenamiento:')
print(target_train.value_counts(normalize=True))

print('\nConjunto de validación:')
print(features_valid.shape)
print(target_valid.shape)
print('\nDistribución del conjunto de validación:')
print(target_valid.value_counts(normalize=True))

print('\nConjunto de prueba:')
print(features_test.shape)
print(target_test.shape)
print('\nDistribución del conjunto de prueba:')
print(target_test.value_counts(normalize=True))


Conjunto de entrenamiento:
(1928, 4)
(1928,)
Distribución del conjunto de entrenamiento:
is_ultra
0    0.693465
1    0.306535
Name: proportion, dtype: float64

Conjunto de validación:
(643, 4)
(643,)

Distribución del conjunto de validación:
is_ultra
0    0.693624
1    0.306376
Name: proportion, dtype: float64

Conjunto de prueba:
(643, 4)
(643,)

Distribución del conjunto de prueba:
is_ultra
0    0.693624
1    0.306376
Name: proportion, dtype: float64


Las características (calls, minutes, messages y mb_used) se separaron de la variable objetivo (is_ultra). Los datos se dividieron en conjuntos de entrenamiento, validación y prueba utilizando una proporción 60/20/20.

Se utilizó estratificación durante la división para mantener aproximadamente la proporción original de usuarios Smart y Ultra en los tres conjuntos. El conjunto de entrenamiento se empleará para ajustar los modelos, el de validación para comparar su desempeño y seleccionar los hiperparámetros, mientras que el conjunto de prueba permanecerá reservado hasta haber seleccionado el modelo final.


## Entrenar los diferentes modelos

Se comenzará evaluando un modelo de árbol de decisión. Se probarán diferentes valores de max_depth para determinar qué nivel de complejidad ofrece la mejor exactitud en el conjunto de validación. Una profundidad demasiado baja puede provocar subajuste, mientras que una profundidad excesiva puede hacer que el modelo se sobreajuste a los datos de entrenamiento.

In [7]:
# Evaluar diferentes profundidades para el árbol de decisión
best_tree_model = None
best_tree_score = 0
best_tree_depth = 0
for depth in range(1, 21): # prueba valores de max_depth de 1 a 20
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train) # entrena el modelo en el conjunto de entrenamiento
    score = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación
    print(f"max_depth={depth} \tscore={score:.4f}")
    if score > best_tree_score:
        best_tree_model = model
        best_tree_score = score
        best_tree_depth = depth

print(f"La exactitud del mejor modelo en el conjunto de validación (max_depth = {best_tree_depth}): {best_tree_score}")

max_depth=1 	score=0.7403
max_depth=2 	score=0.7729
max_depth=3 	score=0.7776
max_depth=4 	score=0.7543
max_depth=5 	score=0.7854
max_depth=6 	score=0.7745


max_depth=7 	score=0.7869
max_depth=8 	score=0.8025
max_depth=9 	score=0.7823
max_depth=10 	score=0.7729
max_depth=11 	score=0.7621
max_depth=12 	score=0.7527
max_depth=13 	score=0.7449
max_depth=14 	score=0.7558
max_depth=15 	score=0.7481
max_depth=16 	score=0.7387
max_depth=17 	score=0.7154
max_depth=18 	score=0.7076
max_depth=19 	score=0.7045
max_depth=20 	score=0.7061
La exactitud del mejor modelo en el conjunto de validación (max_depth = 8): 0.80248833592535


La exactitud del modelo aumenta conforme se incrementa la profundidad del árbol hasta alcanzar su mejor resultado con max_depth=8, donde obtiene una exactitud de aproximadamente 0.8025 en el conjunto de validación. A partir de este punto se observa una tendencia general descendente, aunque con algunas variaciones y pequeñas recuperaciones en profundidades posteriores. Esto es consistente con un aumento de la complejidad del modelo: al generar reglas cada vez más específicas para el conjunto de entrenamiento, el árbol puede ajustarse también a particularidades o ruido de esos datos, reduciendo su capacidad para generalizar de forma estable a nuevas observaciones.

Por esta razón, se selecciona max_depth=8 como la mejor profundidad entre los valores evaluados. Además, su exactitud de aproximadamente 0.80 supera el umbral mínimo de 0.75 establecido para el proyecto.

A continuación, se evaluará un modelo de bosque aleatorio. A diferencia de un árbol de decisión individual, este algoritmo combina las predicciones de múltiples árboles, lo que puede producir un modelo más robusto y reducir el efecto de las particularidades aprendidas por un solo árbol. Se probarán diferentes combinaciones del número de árboles (n_estimators) y su profundidad máxima (max_depth) para encontrar la configuración con mayor exactitud en el conjunto de validación.

In [8]:
# Evaluar diferentes configuraciones para el bosque aleatorio
best_forest_model = None
best_forest_score = 0
best_forest_est = 0
best_forest_depth = 0

for est in range(10, 101, 10):
    best_score = 0
    best_depth = 0
    for depth in range(1, 21):
        model = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth=depth )
        model.fit(features_train, target_train) # entrena el modelo en el conjunto de entrenamiento
        score = model.score(features_valid, target_valid) # calcula la puntuación de accuracy en el conjunto de validación
        if score > best_forest_score:
            best_forest_model = model
            best_forest_score = score
            best_forest_est = est
            best_forest_depth = depth
        if score > best_score:
            best_score = score
            best_depth = depth
    print(f"n_estimators={est} \tmax_depth={best_depth} \tscore={best_score:.4f}")

print(f"La exactitud del mejor modelo en el conjunto de validación (n_estimators = {best_forest_est}, max_depth = {best_forest_depth}): {best_forest_score}")

n_estimators=10 	max_depth=6 	score=0.8040
n_estimators=20 	max_depth=9 	score=0.8134
n_estimators=30 	max_depth=8 	score=0.8196
n_estimators=40 	max_depth=9 	score=0.8212
n_estimators=50 	max_depth=9 	score=0.8180
n_estimators=60 	max_depth=9 	score=0.8149
n_estimators=70 	max_depth=12 	score=0.8180
n_estimators=80 	max_depth=12 	score=0.8196
n_estimators=90 	max_depth=10 	score=0.8165
n_estimators=100 	max_depth=8 	score=0.8149
La exactitud del mejor modelo en el conjunto de validación (n_estimators = 40, max_depth = 9): 0.8211508553654744



El modelo de bosque aleatorio alcanzó su mejor resultado con n_estimators=40 y max_depth=9, obteniendo una exactitud aproximada de 0.8212 en el conjunto de validación. Este resultado supera tanto el umbral mínimo de 0.75 establecido para el proyecto como la exactitud obtenida previamente con el árbol de decisión (0.8025).

Los resultados muestran que incrementar el número de árboles no produce necesariamente una mejora continua en la exactitud. A partir de determinadas configuraciones, el desempeño se mantiene relativamente estable y presenta pequeñas variaciones. De manera similar, una mayor profundidad no garantiza un mejor resultado, por lo que la selección conjunta de n_estimators y max_depth resulta importante para encontrar un equilibrio entre capacidad del modelo y generalización.

El modelo supera el umbral mínimo de exactitud de 0.75 establecido para el proyecto. Hasta este punto, el bosque aleatorio es el modelo con mejor desempeño.

Finalmente, se evaluará un modelo de regresión logística. A diferencia de los modelos basados en árboles, este algoritmo utiliza las características para estimar la probabilidad de pertenecer a cada clase. Se utilizará el solver liblinear, empleado durante las lecciones del curso y adecuado para problemas de clasificación binaria con conjuntos de datos relativamente pequeños. El solver es el algoritmo de optimización encargado de encontrar los coeficientes del modelo.

In [9]:
# Entrenar modelo de regresión logística
logistic_model = LogisticRegression(random_state=12345, solver='liblinear')

logistic_model.fit(features_train, target_train)

logistic_score = logistic_model.score(features_valid, target_valid)

print(f"Exactitud del modelo de regresión logística en el conjunto de validación: {logistic_score:.4f}")

Exactitud del modelo de regresión logística en el conjunto de validación: 0.7185


El modelo de regresión logística obtuvo una exactitud de aproximadamente 0.7185 en el conjunto de validación. Este resultado se encuentra por debajo del umbral mínimo de 0.75 establecido para el proyecto y también es inferior a los resultados obtenidos con el árbol de decisión (0.8025) y el bosque aleatorio (0.8212).

Por lo tanto, entre los modelos evaluados hasta este punto, el bosque aleatorio con n_estimators=40 y max_depth=9 presenta la mayor exactitud en el conjunto de validación.

## Comprobación del modelo con el conjunto de prueba

Después de comparar los modelos en el conjunto de validación, el bosque aleatorio con n_estimators=40 y max_depth=9 obtuvo la mayor exactitud, con un valor aproximado de 0.8212. Por esta razón, se seleccionó como modelo final.

A continuación, se evaluará su desempeño utilizando el conjunto de prueba. Esto permitirá estimar la capacidad del modelo para generalizar a observaciones que no fueron utilizadas durante su desarrollo.

In [10]:
# Evaluar el modelo seleccionado en el conjunto de prueba

test_score = best_forest_model.score(features_test, target_test)

print(f"Exactitud del modelo en el conjunto de prueba: {test_score:.4f}")

Exactitud del modelo en el conjunto de prueba: 0.8087



El modelo de bosque aleatorio seleccionado, con n_estimators=40 y max_depth=9, obtuvo una exactitud de aproximadamente 0.8087 en el conjunto de prueba. Este resultado supera el umbral mínimo de 0.75 establecido para el proyecto.

La exactitud obtenida en prueba es ligeramente menor que la alcanzada en validación, donde el modelo obtuvo aproximadamente 0.8212, pero la diferencia es pequeña. Esto sugiere que el modelo mantiene un desempeño estable al trabajar con observaciones que no fueron utilizadas durante el entrenamiento ni durante la selección de hiperparámetros.

Por lo tanto, el bosque aleatorio presenta una capacidad de generalización adecuada para esta tarea y cumple con el objetivo de clasificación planteado.


## Prueba de cordura

Para comprobar que el modelo seleccionado realmente aporta capacidad predictiva, se comparará su exactitud con la de un modelo de referencia simple. Debido a que la clase Smart (0) representa aproximadamente el 69 % de las observaciones, un clasificador que predijera siempre la clase mayoritaria podría alcanzar una exactitud cercana a 0.69 sin utilizar ninguna de las características de comportamiento.

In [11]:
# Prueba de cordura: predecir siempre la clase mayoritaria (Smart = 0)
predictions = pd.Series(0, index=target_test.index)
score = accuracy_score(target_test, predictions)

print(f"Exactitud del modelo de referencia: {score:.4f}")

Exactitud del modelo de referencia: 0.6936


Como prueba de cordura, se comparó el bosque aleatorio con una estrategia de referencia que predice siempre la clase mayoritaria, correspondiente al plan Smart (0). Esta estrategia alcanzó una exactitud de 0.6936, equivalente aproximadamente a la proporción de usuarios Smart en el conjunto de prueba. El bosque aleatorio obtuvo una exactitud de 0.8087, superando la referencia en aproximadamente 11.5 puntos porcentuales. Por lo tanto, el modelo supera una estrategia trivial de clasificación y pasa la prueba de cordura.

## Conclusión general

En este proyecto se desarrolló un modelo de clasificación para recomendar a los clientes de Megaline uno de sus planes actuales, Smart o Ultra, a partir de su comportamiento mensual. El conjunto de datos contenía 3214 observaciones y cuatro características relacionadas con el uso del servicio: número de llamadas, minutos utilizados, mensajes enviados y tráfico de Internet. No se encontraron valores ausentes ni registros duplicados, por lo que los datos pudieron utilizarse directamente para el modelado.

La variable objetivo presentó un desequilibrio moderado entre las clases, con aproximadamente un 69 % de usuarios en el plan Smart y un 31 % en Ultra. Para conservar esta distribución, los datos se dividieron de forma estratificada en 60 % para entrenamiento, 20 % para validación y 20 % para prueba.

Se evaluaron tres algoritmos de clasificación. El árbol de decisión alcanzó su mejor resultado con max_depth=8, obteniendo una exactitud de aproximadamente 0.8025 en validación. El bosque aleatorio mejoró este resultado y alcanzó una exactitud de 0.8212 con n_estimators=40 y max_depth=9. Por su parte, la regresión logística obtuvo una exactitud de 0.7185, por debajo del umbral mínimo de 0.75 establecido para el proyecto.

Con base en estos resultados, se seleccionó el bosque aleatorio como modelo final. Al evaluarlo con el conjunto de prueba, que no había sido utilizado durante el entrenamiento ni la selección de hiperparámetros, obtuvo una exactitud de 0.8087. La pequeña diferencia respecto al resultado de validación indica que el modelo mantiene un desempeño relativamente estable con datos nuevos y supera el objetivo mínimo planteado.

Finalmente, se realizó una prueba de cordura utilizando como referencia una estrategia que siempre predice la clase mayoritaria, Smart. Esta estrategia obtuvo una exactitud de 0.6936, mientras que el bosque aleatorio alcanzó 0.8087, una mejora aproximada de 11.5 puntos porcentuales. Por lo tanto, el modelo no solo supera el umbral requerido, sino que también ofrece una capacidad predictiva claramente superior a una estrategia de referencia que no utiliza las características de comportamiento de los usuarios.

En conclusión, el Random Forest con 40 árboles y una profundidad máxima de 9 fue el modelo con mejor desempeño entre las configuraciones evaluadas y representa una solución adecuada para la tarea planteada de recomendar los planes Smart o Ultra a los clientes de Megaline.